# Task #4a: Comment-Level Text Features

Text feature extraction for the standardized Coarse Discourse corpus, using
`artifacts/text_features.py`. One row per comment (utterance), computed on
`text_clean`.

| feature | meaning |
|---|---|
| `word_count` | whitespace-separated tokens |
| `char_count` | characters |
| `has_question` | contains `?` |
| `exclaim_count` | number of `!` |
| `caps_ratio` | uppercase letters / all letters (0 when no letters) |
| `has_quote` | any line starts with `>` (Reddit quote) |
| `has_url` | contains `http` |
| `is_deleted` | source `[deleted]` / `[removed]` marker |

Synthetic placeholder utterances are skipped, as Task #3 recommends. The notebook
is aggregate-only: raw text and speaker handles are never displayed.

To run this yourself, download the standardized dataset from Google Drive into
`data/local/` as described in `data/STANDARDIZATION.md`.

# Summary

- Features extracted for all 115,827 non-synthetic Coarse Discourse comments,
  matching the manifest, with no missing values.
- Comments are short: median 22 words, but right-tailed (99th percentile 340 words).
- 21% of comments contain `?`, 8% contain a URL, 2% quote another comment, and
  1% are deleted.
- `word_count` and `char_count` are almost perfectly correlated (0.98), so only
  one is needed for modeling.
- Lengths and `exclaim_count` are heavily skewed; log-transform them before modeling.
- `has_url` matches any text containing `http`, so it can over-count slightly.

# Approach

**Hand-crafted lexical features only.** They are cheap to compute, easy to
interpret, need no extra dependencies, and match the patterns found in Task #3.
TF-IDF and sentence embeddings were not added here because Task 5 already
handles text representation for its model.

## 0. Setup

In [2]:
import hashlib
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

current = Path.cwd().resolve()
REPO_ROOT = next(
    path
    for path in [current, *current.parents]
    if (path / "pyproject.toml").exists()
)
sys.path.insert(0, str(REPO_ROOT / "artifacts"))

from standardize_convokit import iter_standardized
from text_features import extract_features

data_candidates = [
    REPO_ROOT / "data" / "local" / "standardized-v1-regenerated",
    REPO_ROOT / "data" / "local" / "standardized-v1",
]
DATA_DIR = next(path for path in data_candidates if path.exists())
COARSE_PATH = DATA_DIR / "reddit-coarse-discourse-corpus.jsonl"

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")
plt.style.use("seaborn-v0_8-whitegrid")

print(f"Repository: {REPO_ROOT}")
print(f"Dataset:    {DATA_DIR}")

Repository: C:\Users\ravje\Market-Research-1B-discussion-intelligence-toolkit
Dataset:    C:\Users\ravje\Market-Research-1B-discussion-intelligence-toolkit\data\local\standardized-v1


## 1. Validate the local handoff

Same check as Task #3: the JSONL file must match the hash in its manifest.

In [3]:
def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


manifest = json.loads(COARSE_PATH.with_suffix(".manifest.json").read_text(encoding="utf-8"))
actual_hash = sha256(COARSE_PATH)
assert actual_hash == manifest["output"]["sha256"], "Checksum mismatch"
print(f"Coarse Discourse: checksum verified ({actual_hash[:12]}…)")

Coarse Discourse: checksum verified (3656e57ce889…)


## 2. Build the feature table

In [4]:
all_utterances = []
for conversation in iter_standardized(COARSE_PATH):
    all_utterances.extend(conversation.utterances)

features = extract_features(all_utterances)
print("feature table:", features.shape)

feature table: (115827, 10)


In [5]:
# sanity checks
FEATURES = [
    "word_count", "char_count", "has_question", "exclaim_count",
    "caps_ratio", "has_quote", "has_url", "is_deleted",
]

assert len(features) == manifest["counts"]["source_utterances"]
assert features["utterance_id"].is_unique
assert features[FEATURES].notna().all().all()
assert features["caps_ratio"].between(0, 1).all()
assert (features["word_count"] <= features["char_count"]).all()

print("row count matches manifest, ids unique, no nulls, caps_ratio in [0, 1]")
features.drop(columns=["utterance_id", "conversation_id"]).head()

row count matches manifest, ids unique, no nulls, caps_ratio in [0, 1]


,word_count,char_count,has_question,exclaim_count,caps_ratio,has_quote,has_url,is_deleted
0,100,543,True,0,0.026,False,False,False
1,69,365,False,0,0.000,False,False,False
2,23,107,True,0,0.085,False,False,False
3,54,297,False,0,0.029,False,False,False
4,27,121,False,0,0.000,False,False,False


## 3. Feature distributions

Numeric features are summarised with percentiles because they are right-skewed.
Boolean features are shown as rates.

In [6]:
numeric_features = ["word_count", "char_count", "exclaim_count", "caps_ratio"]
boolean_features = ["has_question", "has_quote", "has_url", "is_deleted"]

features[numeric_features].describe(percentiles=[0.5, 0.9, 0.99]).T

,count,mean,std,min,50%,90%,99%,max
word_count,"115,827.000",43.123,76.662,0.000,22.000,97.000,340.000,"3,862.000"
char_count,"115,827.000",242.295,439.832,0.000,121.000,543.000,"1,965.740","22,107.000"
exclaim_count,"115,827.000",0.200,1.291,0.000,0.000,1.000,3.000,298.000
caps_ratio,"115,827.000",0.058,0.098,0.000,0.037,0.110,0.470,1.000


In [7]:
features[boolean_features].mean().rename("rate").to_frame()

,rate
has_question,0.211
has_quote,0.023
has_url,0.078
is_deleted,0.009


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for axis, column in zip(axes, ["word_count", "exclaim_count", "caps_ratio"]):
    values = features[column]
    clipped = values[values <= values.quantile(0.99)]
    axis.hist(clipped, bins=50, edgecolor="white")
    axis.set_title(column)
    axis.set_xlabel(f"{column} (≤ p99)")
    axis.set_ylabel("Comments")

plt.tight_layout()
plt.show()

In [ ]:
features[["word_count", "char_count"]].corr()

Notes:
* Median comment is 22 words; the distribution is right-tailed (p99 340 words)
* Most comments have no `!` and a low `caps_ratio` (median ~4%); the tail is short
  all-caps replies
* 21% of comments contain `?`, 8% contain a URL, 2% quote another comment, 1% are deleted
* `word_count` and `char_count` are 0.98 correlated, so keep only one for modeling

## 4. Save output

Written to Git-ignored `data/local/`; share it through the team Drive, not Git.

In [ ]:
output_path = REPO_ROOT / "data" / "local" / "text_features.csv"
features.to_csv(output_path, index=False)
print(f"saved {len(features):,} rows to {output_path}")